#Distributed Sales Analytics using PySpark RDDs

###Objective is to use PySpark RDDs to load this data, join it, and find the top 3 sales persons globally based on their aggregated sales figures, while also calculating their total sales region-wise.

### Made By - [Ranadip Manna]()

1. Initialization and Data Setup

In [1]:
from pyspark import SparkContext
import sys

In [2]:
# Initialize Spark Context
sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")

In [3]:
# Sample Data Strings (As per assessment requirements)
sales_people_data = """id,name,region
101,Alice Johnson,North
102,Bob Smith,South
103,Charlie Brown,East
104,Diana Prince,West
105,Eve Adams,North
106,Frank Castle,South
107,Grace Hopper,East
108,Hank Pym,West
109,Ivy Pepper,North
110,Jack Ross,South"""

# Expanded Sales Data (20 Transactions)
sales_data_data = """sale_id,person_id,amount
1,101,1500.50
2,102,800.00
3,101,2200.75
4,103,300.25
5,104,4500.00
6,105,1000.00
7,102,1200.00
8,104,500.50
9,101,100.00
10,105,500.00
11,106,2500.00
12,107,1200.50
13,108,3000.00
14,109,450.75
15,110,1800.00
16,106,500.25
17,108,1500.00
18,101,400.00
19,104,1200.00
20,107,800.00"""

In [4]:
with open("sales_people.csv", "w") as f:
    f.write(sales_people_data)

with open("sales_data.csv", "w") as f:
    f.write(sales_data_data)

print("Files 'sales_people.csv' and 'sales_data.csv' created successfully.")

Files 'sales_people.csv' and 'sales_data.csv' created successfully.


2. Loading and Pre-processing (Map Phase)

In [5]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

# Load files as RDDs
people_rdd_raw = sc.textFile("sales_people.csv")
sales_rdd_raw = sc.textFile("sales_data.csv")

In [6]:
# Remove headers
people_header = people_rdd_raw.first()
sales_header = sales_rdd_raw.first()

In [7]:
# Map People: (person_id, (name, region))
people_mapped = (people_rdd_raw
                 .filter(lambda line: line != people_header)
                 .map(lambda line: line.split(","))
                 .map(lambda cols: (int(cols[0]), (cols[1], cols[2]))))

In [8]:
# Map Sales: (person_id, amount)
sales_mapped = (sales_rdd_raw
                .filter(lambda line: line != sales_header)
                .map(lambda line: line.split(","))
                .map(lambda cols: (int(cols[1]), float(cols[2]))))

3. Aggregation and Join (Reduce Phase)

In [9]:
sales_totals = sales_mapped.reduceByKey(lambda a, b: a + b)

joined_data = people_mapped.join(sales_totals)

In [10]:
final_sales_rdd = joined_data.map(lambda x: (x[1][0][0], x[1][0][1], x[1][1])).cache()

4. Final Results

In [11]:
# Objective 1: Top 3 Global Sales Persons
top_3 = final_sales_rdd.takeOrdered(3, key=lambda x: -x[2])

print("\n--- TOP 3 GLOBAL SALES PERSONS ---")
for i, (name, region, total) in enumerate(top_3, 1):
    print(f"{i}. {name} | Region: {region} | Total Sales: {total}")


--- TOP 3 GLOBAL SALES PERSONS ---
1. Diana Prince | Region: West | Total Sales: 6200.5
2. Hank Pym | Region: West | Total Sales: 4500.0
3. Alice Johnson | Region: North | Total Sales: 4201.25


In [12]:
# Objective 2: Total Sales Region-wise
region_wise = (final_sales_rdd
               .map(lambda x: (x[1], x[2]))
               .reduceByKey(lambda a, b: a + b)
               .collect())

print("\n--- REGION-WISE TOTAL SALES ---")
for region, total in region_wise:
    print(f"{region}: {total}")


--- REGION-WISE TOTAL SALES ---
West: 10700.5
North: 6152.0
South: 6800.25
East: 2300.75


In [13]:
# Stop Spark
sc.stop()